<a href="https://colab.research.google.com/github/TimofeyProtasov/diploma/blob/main/days/work_1704%2B%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q triton trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 51.7 MB/s eta 0:00:00


In [33]:
import re
import torch
import random
import numpy as np
from typing import Optional
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from dataclasses import dataclass, field
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from trl import SFTTrainer, SFTConfig
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftConfig
)
from tqdm import tqdm

In [42]:
@dataclass
class RAGExperiment:
    dataset_name: str = 'phatvo/hotpotqa-raft'
    oracle_presence_ratio: float = 1.0
    sample_size: int = 10
    test_size_ratio: float = 0.2
    model_name: str = 'Qwen/Qwen2.5-0.5B-Instruct'
    peft_config: Optional[PeftConfig] = field(default_factory=lambda: LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ))
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 2
    learning_rate: float = 5e-5
    warmup_steps: int = 5
    seed: int = 1


    def __post_init__(self):
        random.seed(self.seed)


    def prepare_data(self):
        dataset: DatasetDict = load_dataset('phatvo/hotpotqa-raft')


        def presence_oracle(example):
            return example['oracle_context'] in example['instruction']


        dataset_with_oracle: Dataset = dataset['train'].filter(lambda ex: presence_oracle(ex))
        dataset_without_oracle: Dataset = dataset['train'].filter(lambda ex: not presence_oracle(ex))

        num_with: int = round(self.sample_size * self.oracle_presence_ratio)
        num_without: int = self.sample_size - num_with

        indices_with: list = random.sample(range(len(dataset_with_oracle)), num_with)
        indices_without: list = random.sample(range(len(dataset_without_oracle)), num_without)

        if indices_with:
            idx_train_with_oracle, idx_test_with_oracle = train_test_split(
                indices_with,
                test_size=self.test_size_ratio,
                shuffle=False,
                random_state=self.seed
            )
        else:
            idx_train_with_oracle, idx_test_with_oracle = [], []

        if indices_without:
            idx_train_without_oracle, idx_test_without_oracle = train_test_split(
                indices_without,
                test_size=self.test_size_ratio,
                shuffle=False,
                random_state=self.seed
            )
        else:
            idx_train_without_oracle, idx_test_without_oracle = [], []

        def get_shuffle_dataset(ids_with, ids_without):
            parts = []
            if ids_with:
                parts.append(dataset_with_oracle.select(ids_with))
            if ids_without:
                parts.append(dataset_without_oracle.select(ids_without))
            if parts:
                return concatenate_datasets(parts).shuffle(seed=self.seed)
            else:
                return Dataset.from_dict({})


        self.train_dataset: Dataset = get_shuffle_dataset(idx_train_with_oracle, idx_train_without_oracle)
        self.test_dataset: Dataset = get_shuffle_dataset(idx_test_with_oracle, idx_test_without_oracle)


    def prepare_model(self):
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            dtype=torch.bfloat16,
            device_map='auto'
        )


    def prepare_tokenizer(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name
        )


    def format_data(self, dataset):
        """Форматируем данные в prompt/completion для SFTTrainer."""
        def format_sft_example(example):
            prompt_messages = [{"role": "user", "content": example["instruction"]}]
            prompt = self.tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )
            completion = example["cot_answer"]
            return {"prompt": prompt, "completion": completion}

        return dataset.map(
            format_sft_example,
            remove_columns=dataset.column_names
        )


    def add_peft_if_exist(self):
        self.model = get_peft_model(self.model, self.peft_config) if self.peft_config else self.model


    def prepare_training(self):
        self.training_args = SFTConfig(
            output_dir="./qwen-raft-lora",
            num_train_epochs=self.num_train_epochs,
            per_device_train_batch_size=self.per_device_train_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            warmup_steps=self.warmup_steps,
            logging_steps=10,
            save_steps=100,
            save_total_limit=2,
            bf16=True,
            report_to="none",
            remove_unused_columns=False,

            max_length=1024,
            packing=False,
            completion_only_loss=True,
        )

        self.trainer = SFTTrainer(
            model=self.model,
            args=self.training_args,
            train_dataset=self.formatted_train_dataset,
            processing_class=self.tokenizer,
        )


    def train(self):
        self.trainer.train()

    def extract_answer(self, text: str) -> str:
        match = re.search(r"<ANSWER>:\s*(.*?)(?:\n|$)", text, re.DOTALL)
        if match:
            return match.group(1).strip()
        # fallback — возвращаем весь текст
        return text.strip()

    def token_level_f1(self, pred: str, true: str) -> float:
        """Вычисляет F1 по токенам (словам) как пересечение множеств."""
        if not pred or not true:
            return 0.0

        pred_tokens = set(pred.lower().split())
        true_tokens = set(true.lower().split())
        common = pred_tokens & true_tokens

        if not common:
            return 0.0

        precision = len(common) / len(pred_tokens)
        recall = len(common) / len(true_tokens)
        f1 = 2 * precision * recall / (precision + recall)
        return f1

    def evaluate_f1(self, debug_examples: int = 3):
        """
        Вычисляет среднюю F1-меру на тестовом наборе.
        Для первых `debug_examples` примеров выводит полную информацию.
        """
        self.model.eval()
        formatted_test = self.format_data(self.test_dataset)
        f1_scores = []

        for i, example in enumerate(tqdm(formatted_test, desc="Evaluating F1")):
            prompt = example["prompt"]
            true_completion = example["completion"]
            true_answer = self.extract_answer(true_completion)

            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=2048,   # увеличенный лимит для длинных рассуждений
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,  # явно указываем EOS
                )

            # Получаем полный сгенерированный текст (промпт + продолжение)
            full_output = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            generated_part = full_output[len(prompt):].strip()

            # Извлекаем предсказанный ответ (если нет тега, вернёт весь сгенерированный текст)
            pred_answer = self.extract_answer(generated_part)

            # --- ОТЛАДОЧНЫЙ ВЫВОД ---
            if i < debug_examples:
                print("\n" + "="*80)
                print(f"📋 ПРИМЕР {i}")
                print("="*80)

                print("\n🔹 ПОЛНЫЙ ПРОМПТ (с chat-шаблоном):")
                print("-" * 40)
                print(prompt)
                print("-" * 40)

                print("\n🔹 ЭТАЛОННЫЙ COMPLETION (первые 300 символов):")
                print("-" * 40)
                print(true_completion[:300] + ("..." if len(true_completion) > 300 else ""))
                print("-" * 40)

                print("\n🔹 ЭТАЛОННЫЙ ОТВЕТ (извлечённый):")
                print("-" * 40)
                print(true_answer)
                print("-" * 40)

                print("\n🔹 СГЕНЕРИРОВАННАЯ ЧАСТЬ (первые 500 символов):")
                print("-" * 40)
                print(generated_part + ("..." if len(generated_part) > 500 else ""))
                print("-" * 40)

                print("\n🔹 ПРЕДСКАЗАННЫЙ ОТВЕТ (извлечённый):")
                print("-" * 40)
                print(pred_answer)
                print("-" * 40)

                # Проверяем, содержится ли тег <ANSWER> в сгенерированном тексте
                has_tag = "<ANSWER>" in generated_part
                print(f"\n🔸 Наличие тега <ANSWER> в генерации: {'✅ ДА' if has_tag else '❌ НЕТ'}")

                # Показываем, что модель сгенерировала в самом конце (последние 200 символов)
                print("\n🔹 ПОСЛЕДНИЕ 200 СИМВОЛОВ ГЕНЕРАЦИИ:")
                print("-" * 40)
                print(generated_part[-200:] if len(generated_part) > 200 else generated_part)
                print("-" * 40)

            # Вычисляем F1
            f1 = self.token_level_f1(pred_answer, true_answer)
            f1_scores.append(f1)

        avg_f1 = np.mean(f1_scores) if f1_scores else 0.0
        print("\n" + "="*80)
        print(f"✅ Средняя F1 по всем {len(f1_scores)} примерам: {avg_f1:.4f}")
        print("="*80)
        return avg_f1


    def evaluate_perplexity(self):
        pass


    def run(self):
        self.prepare_data()
        self.prepare_model()
        self.prepare_tokenizer()
        self.formatted_train_dataset = self.format_data(self.train_dataset)
        self.add_peft_if_exist()
        self.prepare_training()
        self.train()
        self.evaluate_f1()
        self.evaluate_perplexity()
        torch.cuda.empty_cache()

In [48]:
exp = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=300,
    test_size_ratio=0.03,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [49]:
exp.run()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/291 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/291 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/291 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


...


📋 ПРИМЕР 1

🔹 ПОЛНЫЙ ПРОМПТ (с chat-шаблоном):
----------------------------------------
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
<DOCUMENT>The 2012 Summer Olympics, officially known as the Games of the XXX Olympiad, were held in London, United Kingdom, from 25 July 2012 to 12 August 2012.</DOCUMENT>
<DOCUMENT>Capleville is an unincorporated community in Shelby County, Tennessee, United States, southeast of Memphis and just north of the Mississippi border. It is located 0.5 mi. east of the Memphis International Airport, starting 1 mi. west of the intersection of State Routes 176 and 175, and heading east along State Route 175 (Shelby Drive) crossing U.S. Route 78. Most of the area has been incorporated into the City of Memphis and since has become a large industrial center due to its proximity to the airport and Lamar Avenue (U.S. Route 78) which becomes a divided freeway after State Route 175.</DOCUMENT>
<DOCUME

Evaluating F1:  33%|███▎      | 3/9 [00:22<00:45,  7.66s/it]


📋 ПРИМЕР 2

🔹 ПОЛНЫЙ ПРОМПТ (с chat-шаблоном):
----------------------------------------
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
<DOCUMENT>Andries Bonger (20 May 1861 – 20 January 1936), nicknamed "Dries", was Johanna van Gogh-Bonger's favorite brother. Bonger was a friend of his future brother-in-law Theo van Gogh in Paris. It was through Andries that Johanna and Theo met. He also knew Vincent van Gogh who called him André in letters.</DOCUMENT>
<DOCUMENT>The Master of Nuremberg (German: Der Meister von Nürnberg) is a 1927 German silent historical comedy film directed by Ludwig Berger and starring Rudolf Rittner, Max Gülstorff and Gustav Fröhlich. It is based on the 1868 opera "Die Meistersinger von Nürnberg" by Richard Wagner. It was considered artistically unsuccessful because of its overly theatrical presentation. It is also known by the alternative title The Meistersinger.</DOCUMENT>
<DOCUMENT>Daniel Edward

Evaluating F1: 100%|██████████| 9/9 [01:06<00:00,  7.41s/it]


✅ Средняя F1 по всем 9 примерам: 0.6338
